# 외부 기사 이진 분류 — KLUE-RoBERTa 모델 예측

**목적**: 학습에 사용된 적 없는 실제 뉴스 기사 11건을 모델에 입력하여, 모델이 스스로 낚시성/정상을 판단하는 결과 확인  
**모델**: `klue_binary_final.pt` (전체 데이터 재학습본)  
**방식**: 정답 라벨 없음 — 모델이 각 기사를 낚시성(1) 또는 정상(0)으로 판단한 결과와 확률만 출력  
**샘플**: 실제 뉴스 기사 11건 (2026년 5~6월)

In [ ]:
import os, torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BASE_DIR   = os.getcwd()
MODEL_NAME = 'klue/roberta-base'
BIN_PT     = os.path.join(BASE_DIR, 'klue_binary_final.pt')
MAX_LEN    = 128
LABEL      = {0: '정상', 1: '낚시성'}

print('설정 완료')
print(f'모델 파일: {BIN_PT}')
print(f'파일 존재 여부: {os.path.exists(BIN_PT)}')

In [ ]:
print('토크나이저 및 모델 로딩 중...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.load_state_dict(torch.load(BIN_PT, map_location='cpu'))
model.eval()
print('로딩 완료!')

In [ ]:
# 실제 수집 기사 11건 (제목, 본문 앞부분, 출처)
# 정답 라벨 없음: 모델이 직접 낚시성/정상을 판단
SAMPLES = [
    ('"하루 3차례, 비용도 싸다"…예약 폭발한 결혼식 뭐길래',
     '요즘은 남들과 똑같은 결혼식이 아니라 전통혼례를 찾는 예비부부들이 늘고 있다고 합니다. '
     '전통혼례라니 좀 특이하죠. 한복, 전통 소품을 활용한 K-브라이덜 샤워까지 등장했습니다. '
     '이들은 전통을 단순히 재현하는 데 그치지 않고, 부부와 하객 모두가 함께 즐기고 기억할 수 있는 경험으로 재해석하고 있었습니다.',
     'SBS 뉴스'),

    ('"젠슨 황 깐부들도 파랗게 질렸는데"…상한가 찍은 종목 뭐길래',
     '젠슨 황 엔비디아 CEO의 방한을 하루 앞둔 4일 LG전자·네이버 등 관련 수혜주로 주목받아온 종목들이 '
     '차익실현 매물 등장에 일제히 급락했다. 브로드컴이 시장 기대치를 밑돈 매출 전망치를 발표하자 '
     '삼성전자, SK하이닉스 주가가 일제히 약세를 보였다.',
     '중앙일보'),

    ("태안 앞바다서 체포 된 중국인 남성, 알고보니…'충격 정체'",
     '중국 반체제 인사로 알려진 인권운동가 둥광핑(68)이 고무보트를 타고 서해를 건너 '
     '한국 영해로 들어왔다가 충남 태안 앞바다에서 해경에 붙잡혔다. '
     'NYT와 CNN 등 외신은 둥광핑이 소형 고무보트를 타고 한국 영해에 진입했다가 체포됐다고 보도했다.',
     '한국경제'),

    ('사망한 남편, 알고 보니 일본서 두 집 살림…상간녀에 위자료 청구 가능할까',
     '남편이 사망한 이후에 부정행위 사실을 알게 되었더라도 상간녀를 상대로 한 위자료 청구 소송이 가능하다는 법조계 조언이 나왔다. '
     'A씨의 남편은 무역 법인의 중역으로 일본 출장이 잦았다. 지난해 남편이 갑작스러운 심장마비로 세상을 떠나면서 '
     '유품을 정리하던 중 남편의 일본 휴대전화에서 낯선 여성, 아이들과 함께 찍은 사진을 발견했다.',
     '뉴시스'),

    ("'애둘맘' 김보미, 수술 후 병원 갔다가 결국 터졌다…\"간호사 왜 이렇게 불친절\"",
     '배우 김보미가 요로결석 수술 이후 다시 찾은 병원에서 속상했던 심경을 털어놨다. '
     '김보미는 자신의 SNS에 "수술한데 아파서 병원왔는데.. 창구에 있는 간호사 쌤들은 왜이렇게 불친절할까.. '
     '아닌분들도 계시지만.. 하…정말…"이라는 글을 남겼다.',
     'MK스포츠'),

    ("'가지 부부' 아내 결국 터졌다, 남편 향해 \"입 다물어\" (이혼숙려캠프)",
     "JTBC '이혼숙려캠프' 21기 부부들의 최종 조정 과정이 공개된다. "
     "'가지 남편'은 가족보다 가지를 우선으로 생각한다는 검사 결과가 공개돼 충격을 안긴다. "
     "이어진 변호사 상담에서 '가지 남편'이 부부 관계 도중 게임을 한 행동이 명백한 유책 사유라는 사실을 알게 된다.",
     '동아닷컴'),

    ('3% 훌쩍 넘어간 예금금리…증시 활황에도 은행 예치자금 증가',
     '최근 시장금리 상승을 반영한 은행 예금금리가 우상향 곡선을 그리면서 연 3%를 웃돌고 있다. '
     '한국은행이 하반기 기준금리 인상을 예고하면서 앞으로 수신금리도 4%를 넘어설 수 있다는 전망이 나온다. '
     '19개 은행의 1년 만기 정기예금 중 절반이 넘는 상품이 우대금리 포함 최고 3.0% 이상을 제공한다.',
     '뉴시스'),

    ('[팩트체크] 이번 여름 한 달 내내 비온다고?…매년 반복되는 장마 예보 정체는',
     '올해 6~7월에 한 달 내내 비가 내릴 것이란 내용의 게시물이 SNS에서 확산하고 있다. '
     '기상청이 공식 발표가 아니라며 진화에 나섰지만 허위 정보가 횡행한다. '
     '기상청은 기후변화 등의 이유로 2009년 이후 공식 장마 전망을 내놓지 않고 있다.',
     '연합뉴스'),

    ('AI 행정시대, 열린정부 논의…행안부, OECD 국제포럼 개최',
     '행정안전부가 OECD와 해외 정부, 시민사회가 참여하는 국제행사를 열고 '
     'AI 시대 공공거버넌스와 열린정부 방향을 논의한다. '
     "행안부는 서울에서 'OECD 열린정부 국제심포지엄'을 개최한다고 밝혔다.",
     '뉴스1'),

    ('코스피 급락에 매도 사이드카…삼성전자·SK하이닉스 약세',
     '코스피가 5일 급락하면서 프로그램매도호가 일시효력정지인 매도 사이드카가 발동됐다. '
     '한국거래소에 따르면 코스피200선물지수의 변동으로 5분간 프로그램매도호가의 효력이 정지됐다. '
     '발동 시점 코스피200선물지수는 전일 종가보다 71.84포인트(5.20%) 하락했다.',
     '경기일보'),

    ('5월 수출액 877.5억달러 역대 최대…반도체 호황 덕분',
     '5월 수출액이 877억5000만달러로 집계돼 월 기준 역대 최대 기록을 경신했다. '
     '사상 처음으로 월 수출액이 3개월 연속 800억달러를 넘어섰다. '
     'AI 투자 확대에 따라 반도체 호황이 이어진 덕분이다. 수출은 전년 대비 53.2% 증가했다.',
     '조선일보'),
]

print(f'샘플 총 {len(SAMPLES)}건 준비 완료')

In [ ]:
import pandas as pd

rows = []
for title, content, source in SAMPLES:
    enc = tokenizer(
        text=title,
        text_pair=content,
        truncation='only_second',
        max_length=MAX_LEN,
        padding='max_length',
        return_tensors='pt',
    )
    with torch.no_grad():
        out = model(**enc)
    prob = F.softmax(out.logits, dim=-1).squeeze()
    pred = int(torch.argmax(prob))
    rows.append({
        '제목':        title,
        '출처':        source,
        '모델 판정':   LABEL[pred],
        '정상 확률':   f'{prob[0]*100:.2f}%',
        '낚시성 확률': f'{prob[1]*100:.2f}%',
    })

df = pd.DataFrame(rows)
df.index = range(1, len(df)+1)
df

In [ ]:
clickbait_count = sum(1 for r in rows if r['모델 판정'] == '낚시성')
normal_count    = sum(1 for r in rows if r['모델 판정'] == '정상')
total           = len(rows)

print('=' * 50)
print('  모델 판정 요약')
print('=' * 50)
print(f'  전체 기사    : {total}건')
print(f'  낚시성 판정  : {clickbait_count}건')
print(f'  정상 판정    : {normal_count}건')
print('=' * 50)

print('\n  낚시성으로 판정된 기사:')
for r in rows:
    if r['모델 판정'] == '낚시성':
        print(f"  - [{r['출처']}] {r['제목'][:50]}  (낚시: {r['낚시성 확률']})")